# SAM3 Attention Module Discovery

Enumerate all attention modules in the SAM3 image model and group them:

- **Group A** — Prompt/image fusion attention (multimodal, fusion, cross_attn, text_cross_attn)
- **Group B** — Detector query/image attention (decoder layers, cross_attention)

In [1]:
import sys, torch
sys.path.insert(0, '../src')
from pam.sam3_loader import load_sam3_image_model

model, processor = load_sam3_image_model(device='cuda')
print('Model loaded.')

/gandiva/venvs/pam/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Model loaded.


In [2]:
# All attention-related modules
print(f"{'Module Name':<80} {'Type'}")
print('=' * 120)
for name, module in model.named_modules():
    if 'attn' in name.lower():
        print(f'{name:<80} {type(module).__name__}')

Module Name                                                                      Type
backbone.vision_backbone.trunk.blocks.0.attn                                     Attention
backbone.vision_backbone.trunk.blocks.0.attn.qkv                                 Linear
backbone.vision_backbone.trunk.blocks.0.attn.proj                                Linear
backbone.vision_backbone.trunk.blocks.1.attn                                     Attention
backbone.vision_backbone.trunk.blocks.1.attn.qkv                                 Linear
backbone.vision_backbone.trunk.blocks.1.attn.proj                                Linear
backbone.vision_backbone.trunk.blocks.2.attn                                     Attention
backbone.vision_backbone.trunk.blocks.2.attn.qkv                                 Linear
backbone.vision_backbone.trunk.blocks.2.attn.proj                                Linear
backbone.vision_backbone.trunk.blocks.3.attn                                     Attention
backbone.vision_backbo

In [3]:
# Group A: Prompt/image fusion attention
# Keywords: multimodal, fusion, cross_attn (in encoder), text_cross_attn, ca_text
group_a_keywords = ['multimodal', 'fusion', 'encoder.*cross_attn', 'text_cross', 'ca_text',
                     'geometry_encoder.*cross_attn']

import re

print('GROUP A — Prompt / Image Fusion Attention')
print('=' * 120)
for name, module in model.named_modules():
    if 'attn' not in name.lower():
        continue
    if any(re.search(kw, name) for kw in group_a_keywords):
        nparams = sum(p.numel() for p in module.parameters())
        extra = ''
        if hasattr(module, 'num_heads'):
            extra += f'  heads={module.num_heads}'
        if hasattr(module, 'embed_dim'):
            extra += f'  embed_dim={module.embed_dim}'
        if hasattr(module, 'batch_first'):
            extra += f'  batch_first={module.batch_first}'
        print(f'{name:<70} {type(module).__name__:<30} params={nparams:>10,}{extra}')

GROUP A — Prompt / Image Fusion Attention
geometry_encoder.encode.0.cross_attn_image                             MultiheadAttention             params=   263,168  heads=8  embed_dim=256  batch_first=False
geometry_encoder.encode.0.cross_attn_image.out_proj                    NonDynamicallyQuantizableLinear params=    65,792
geometry_encoder.encode.1.cross_attn_image                             MultiheadAttention             params=   263,168  heads=8  embed_dim=256  batch_first=False
geometry_encoder.encode.1.cross_attn_image.out_proj                    NonDynamicallyQuantizableLinear params=    65,792
geometry_encoder.encode.2.cross_attn_image                             MultiheadAttention             params=   263,168  heads=8  embed_dim=256  batch_first=False
geometry_encoder.encode.2.cross_attn_image.out_proj                    NonDynamicallyQuantizableLinear params=    65,792
transformer.encoder.layers.0.cross_attn_image                          MultiheadAttention             para

In [4]:
# Group B: Detector query / image attention
# Keywords: decoder.layers.*.cross_attn (but NOT ca_text)
group_b_keywords = ['decoder.*cross_attn']
group_b_exclude = ['ca_text']

print('GROUP B — Detector Query / Image Attention')
print('=' * 120)
for name, module in model.named_modules():
    if 'attn' not in name.lower():
        continue
    if any(re.search(kw, name) for kw in group_b_keywords) and not any(ex in name for ex in group_b_exclude):
        nparams = sum(p.numel() for p in module.parameters())
        extra = ''
        if hasattr(module, 'num_heads'):
            extra += f'  heads={module.num_heads}'
        if hasattr(module, 'embed_dim'):
            extra += f'  embed_dim={module.embed_dim}'
        if hasattr(module, 'batch_first'):
            extra += f'  batch_first={module.batch_first}'
        print(f'{name:<70} {type(module).__name__:<30} params={nparams:>10,}{extra}')

GROUP B — Detector Query / Image Attention
transformer.decoder.layers.0.cross_attn                                MultiheadAttention             params=   263,168  heads=8  embed_dim=256  batch_first=False
transformer.decoder.layers.0.cross_attn.out_proj                       NonDynamicallyQuantizableLinear params=    65,792
transformer.decoder.layers.1.cross_attn                                MultiheadAttention             params=   263,168  heads=8  embed_dim=256  batch_first=False
transformer.decoder.layers.1.cross_attn.out_proj                       NonDynamicallyQuantizableLinear params=    65,792
transformer.decoder.layers.2.cross_attn                                MultiheadAttention             params=   263,168  heads=8  embed_dim=256  batch_first=False
transformer.decoder.layers.2.cross_attn.out_proj                       NonDynamicallyQuantizableLinear params=    65,792
transformer.decoder.layers.3.cross_attn                                MultiheadAttention             par

In [8]:
# Uncategorised attention modules — summarise by prefix
from collections import Counter

uncat_names = []
for name, module in model.named_modules():
    if 'attn' not in name.lower():
        continue
    if name.endswith('.out_proj'):
        continue
    in_a = any(re.search(kw, name) for kw in group_a_keywords)
    in_b = any(re.search(kw, name) for kw in group_b_keywords) and not any(ex in name for ex in group_b_exclude)
    if not in_a and not in_b:
        uncat_names.append(name)

# Group by top-level prefix
prefixes = Counter()
for n in uncat_names:
    parts = n.split('.')
    # use first 3 components as prefix
    prefix = '.'.join(parts[:min(3, len(parts))])
    prefixes[prefix] += 1

print('UNCATEGORISED — Grouped by prefix')
print('=' * 80)
for prefix, count in prefixes.most_common():
    print(f'  {prefix:<60} count={count}')

print(f'\nTotal uncategorised: {len(uncat_names)}')
print(f'\nSample names (first 5):')
for n in uncat_names[:5]:
    print(f'  {n}')
print(f'...')
print(f'Sample names (last 5):')
for n in uncat_names[-5:]:
    print(f'  {n}')

UNCATEGORISED — Grouped by prefix
  backbone.vision_backbone.trunk                               count=96
  backbone.language_backbone.encoder                           count=24
  transformer.encoder.layers                                   count=6
  transformer.decoder.layers                                   count=6
  geometry_encoder.encode.0                                    count=1
  geometry_encoder.encode.1                                    count=1
  geometry_encoder.encode.2                                    count=1
  segmentation_head.cross_attn_norm                            count=1

Total uncategorised: 136

Sample names (first 5):
  backbone.vision_backbone.trunk.blocks.0.attn
  backbone.vision_backbone.trunk.blocks.0.attn.qkv
  backbone.vision_backbone.trunk.blocks.0.attn.proj
  backbone.vision_backbone.trunk.blocks.1.attn
  backbone.vision_backbone.trunk.blocks.1.attn.qkv
...
Sample names (last 5):
  transformer.decoder.layers.2.self_attn
  transformer.decoder.layers.